# Aing_리그전 Transformer Fine-tune (Fixed Pre-train / FT-only / Test Open)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/)

- Stage A: 고정 Transformer 설정으로 Multi30k 기반 pre-train checkpoint를 준비하거나 로드합니다.
- Stage B: IWSLT fine-tuning을 수행합니다.
- 리그전 튜닝 대상: `FT_HP` 안의 fine-tuning 하이퍼파라미터만 수정합니다.
- 최종 점수: 참가자가 직접 `final` 모드에서 실행한 **IWSLT test BLEU**입니다.

---


### STEP1. 설치 (Colab 기준)
- **목표:** Colab/Jupyter에서 필요한 라이브러리를 설치합니다.
- **왜 필요한가?**
  - `datasets`: HuggingFace에서 Multi30k/IWSLT 같은 데이터셋을 쉽게 불러옵니다.
  - `tokenizers`: BPE(서브워드) 토크나이저를 빠르게 학습/적용합니다.
  - `sacrebleu`: 번역 품질을 수치화하는 **BLEU** 점수를 계산합니다.

- **실행 팁:** 이 셀은 보통 **한 번만** 실행하면 됩니다(런타임 재시작하면 다시 실행).

In [ ]:
!pip -q install -U "transformers==5.2.0" "datasets==4.5.0" "accelerate==1.12.0" "tokenizers==0.22.2" sacrebleu

# 설치 후 import 에러가 나면 런타임 재시작(Runtime > Restart runtime) 후 다시 실행하세요.

### STEP2. import os, math, random, time
- **목표:** 번역 데이터셋을 로드하고(train/valid/test) 필요한 컬럼을 확인합니다.
- **핵심 로직:**
  - `load_dataset(...)` → `DatasetDict`(train/validation/test) 형태로 반환
  - 한 샘플은 보통 `{src_lang: ..., tgt_lang: ...}` 또는 `{'translation': {'de':..., 'en':...}}` 형태
- **용어:**
  - **train**: 학습용(가중치 업데이트)
  - **validation(dev)**: 튜닝/중간 평가용(가중치 업데이트 X)
  - **test**: 최종 평가용(리그전 점수)
- **실험 팁(120분):** train/valid/test 샘플 수를 줄이면 **여러 번 실험**하기 쉬워집니다.

In [ ]:
import os, math, random, time
os.environ["TOKENIZERS_PARALLELISM"] = "false"  # tokenizer 병렬 경고/데드락 방지
from dataclasses import dataclass
from typing import Dict, Any, Optional

import numpy as np
import torch
import sacrebleu

from datasets import load_dataset, Dataset

from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace
from tokenizers.processors import TemplateProcessing

from transformers import (
    PreTrainedTokenizerFast,
    BartConfig,
    BartForConditionalGeneration,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
)

def safe_load_dataset(*args, **kwargs):
    try:
        return load_dataset(*args, **kwargs)
    except Exception as e:
        if 'trust_remote_code' in str(e):
            return load_dataset(*args, trust_remote_code=True, **kwargs)
        raise

# === 공통 유틸: BPE 토크나이저 학습 / HF 토크나이저 래퍼 ===
SPECIAL_TOKENS = ["<pad>", "<bos>", "<eos>", "<unk>"]

def train_bpe_tokenizer(train_ds, vocab_size=8000, min_freq=2, src_key=None, tgt_key=None):
    if src_key is None:
        src_key = globals().get("SRC_KEY", "de")
    if tgt_key is None:
        tgt_key = globals().get("TGT_KEY", "en")

    tok = Tokenizer(BPE(unk_token="<unk>"))
    tok.pre_tokenizer = Whitespace()
    trainer = BpeTrainer(
        vocab_size=vocab_size,
        min_frequency=min_freq,
        special_tokens=SPECIAL_TOKENS,
    )

    def iterator():
        for ex in train_ds:
            yield ex[src_key]
            yield ex[tgt_key]

    tok.train_from_iterator(iterator(), trainer=trainer)

    bos_id = tok.token_to_id("<bos>")
    eos_id = tok.token_to_id("<eos>")
    tok.post_processor = TemplateProcessing(
        single="<bos> $A <eos>",
        special_tokens=[("<bos>", bos_id), ("<eos>", eos_id)],
    )
    return tok

def build_hf_tokenizer_from_tokenizers(tok: Tokenizer) -> PreTrainedTokenizerFast:
    return PreTrainedTokenizerFast(
        tokenizer_object=tok,
        bos_token="<bos>",
        eos_token="<eos>",
        unk_token="<unk>",
        pad_token="<pad>",
    )


### STEP3. def set_seed(seed=42):
- **목표:** 실험이 매번 조금씩 달라지는 현상을 줄이기 위해 **난수 시드(seed)** 를 고정하고, GPU/CPU 디바이스를 설정합니다.
- **핵심 로직:**
  1) `random`, `numpy`, `torch`의 seed를 동일하게 설정
  2) `cuda` 사용 가능하면 GPU로 학습 (속도↑)
- **용어:**
  - **재현성(reproducibility)**: 같은 코드/데이터로 같은 결과가 나오게 하는 성질(리그전에서 중요).

In [ ]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

def make_seq2seq_training_args(**kwargs):
    try:
        return Seq2SeqTrainingArguments(**kwargs)
    except TypeError:
        # 가장 흔한 변경: eval_strategy ↔ evaluation_strategy
        if "eval_strategy" in kwargs and "evaluation_strategy" not in kwargs:
            kwargs["evaluation_strategy"] = kwargs.pop("eval_strategy")
        elif "evaluation_strategy" in kwargs and "eval_strategy" not in kwargs:
            kwargs["eval_strategy"] = kwargs.pop("evaluation_strategy")
        return Seq2SeqTrainingArguments(**kwargs)


### STEP3-1. 리그전 공통 설정

- `LEAGUE_MODE="public_fast"`: 빠른 실험용입니다. validation 일부만 평가하고 beam search를 줄입니다.
- `LEAGUE_MODE="public_full"`: 제출 전 확인용입니다. 전체 validation과 final과 같은 fine-tune 설정을 사용합니다.
- `LEAGUE_MODE="final"`: 참가자가 IWSLT test BLEU까지 직접 계산하는 최종 실행 모드입니다.
- pre-train 모델 구조, tokenizer, pre-train 학습 설정은 고정합니다.
- 참가자는 Stage B의 `FT_HP`만 수정합니다.

`LEAGUE_MODE`는 실험 상황에 맞게 바꿔 사용합니다.
처음 여러 조합을 비교할 때는 `public_fast`, 제출 전 후보를 확인할 때는 `public_full`, 최종 리그전 점수를 계산할 때는 `final`을 사용합니다.


In [ ]:
# === League global settings ===
# 실행 모드
# - public_fast: 여러 조합을 빠르게 비교할 때 사용하는 validation subset 모드
# - public_full: 제출 전 후보 조합을 validation 전체로 확인하는 모드
# - final: 최종 IWSLT test BLEU까지 계산하는 모드
LEAGUE_MODE = "public_fast"  # "public_fast" / "public_full" / "final"

# Stage A: fixed pre-train model setting
# 리그전에서는 아래 모델 구조를 변경하지 않습니다.
PRETRAIN_MODEL_TAG = "fixed_small_transformer"
MODEL_HP = dict(
    d_model=256,
    nhead=4,
    num_layers=3,
    d_ff=1024,
    dropout=0.10,
)

RUN_TEST_EVAL = (LEAGUE_MODE == "final")

# pre-train checkpoint cache 설정
USE_PRETRAINED_CKPT = True
FORCE_RETRAIN_PRETRAIN = False
CACHE_VERSION = "v4_test_open_fixed_pretrain_ft_only"

# 데이터/토크나이저 고정 설정
M30K_TRAIN_SAMPLES_FIXED = 29_000
TOKENIZER_IWSLT_SAMPLES_FIXED = 80_000
IWSLT_SHUFFLE_SEED_FIXED = 42
VOCAB_SIZE_FIXED = 12_000
MIN_FREQ_FIXED = 2
MAX_LEN_FIXED = 80

# Stage A: pre-train 고정 설정
PRETRAIN_MAX_STEPS_FIXED = 1_000
PRETRAIN_LR_FIXED = 5e-4
PRETRAIN_WARMUP_FIXED = 200
PRETRAIN_SCHEDULER_FIXED = "inverse_sqrt"
PRETRAIN_EVAL_STEPS_FIXED = 250
PRETRAIN_BSZ_FIXED = 8
PRETRAIN_ACCUM_FIXED = 4  # effective batch = 32
PRETRAIN_GEN_BEAMS_FIXED = 1

# Stage B: fine-tune 모드별 고정 설정
if LEAGUE_MODE == "public_fast":
    IWSLT_TRAIN_SAMPLES_FIXED = 12_000
    FINETUNE_MAX_STEPS_FIXED = 300
    FINETUNE_EVAL_STEPS_FIXED = 100
    VALID_EVAL_SAMPLES_FIXED = 512
    GEN_BEAMS_FIXED = 1
elif LEAGUE_MODE == "public_full":
    IWSLT_TRAIN_SAMPLES_FIXED = 80_000
    FINETUNE_MAX_STEPS_FIXED = 1_000
    FINETUNE_EVAL_STEPS_FIXED = 200
    VALID_EVAL_SAMPLES_FIXED = None
    GEN_BEAMS_FIXED = 4
elif LEAGUE_MODE == "final":
    IWSLT_TRAIN_SAMPLES_FIXED = 80_000
    FINETUNE_MAX_STEPS_FIXED = 1_000
    FINETUNE_EVAL_STEPS_FIXED = 200
    VALID_EVAL_SAMPLES_FIXED = None
    GEN_BEAMS_FIXED = 4
else:
    raise ValueError('LEAGUE_MODE must be "public_fast", "public_full", or "final"')

FINETUNE_BSZ_FIXED = 8
FINETUNE_ACCUM_FIXED = 4  # effective batch = 32

# Colab Drive가 mount되어 있으면 Drive에, 아니면 현재 작업 디렉터리에 checkpoint를 저장합니다.
DEFAULT_DRIVE_ROOT = "/content/drive/MyDrive/aing_transformer_ckpts"
CHECKPOINT_ROOT = DEFAULT_DRIVE_ROOT if os.path.isdir("/content/drive/MyDrive") else "./aing_transformer_ckpts"
os.makedirs(CHECKPOINT_ROOT, exist_ok=True)

PRETRAIN_CKPT_DIR = os.path.join(
    CHECKPOINT_ROOT,
    f"{CACHE_VERSION}_{PRETRAIN_MODEL_TAG}_vocab{VOCAB_SIZE_FIXED}_len{MAX_LEN_FIXED}"
)

print("LEAGUE_MODE:", LEAGUE_MODE)
print("RUN_TEST_EVAL:", RUN_TEST_EVAL)
print("Fixed MODEL_HP:", MODEL_HP)
print("PRETRAIN_CKPT_DIR:", PRETRAIN_CKPT_DIR)
print("IWSLT_TRAIN_SAMPLES_FIXED:", IWSLT_TRAIN_SAMPLES_FIXED)
print("FINETUNE_MAX_STEPS_FIXED:", FINETUNE_MAX_STEPS_FIXED)
print("VALID_EVAL_SAMPLES_FIXED:", VALID_EVAL_SAMPLES_FIXED)
print("GEN_BEAMS_FIXED:", GEN_BEAMS_FIXED)


## 1) Multi30k/IWSLT 로드 & 토크나이저 준비

- Multi30k는 Stage A pre-train에 사용합니다.
- IWSLT는 Stage B fine-tuning과 validation/test 평가에 사용합니다.
- 최종 리그전 점수는 `final` 모드에서 출력되는 IWSLT test BLEU입니다.


### STEP4. Stage 0: Multi30k/IWSLT 로드
- **목표:** Multi30k와 IWSLT 데이터를 로드합니다.
- **Multi30k:** 고정 pre-train용 데이터
- **IWSLT:** fine-tuning 및 validation/test 평가용 데이터


In [ ]:
multi30k = safe_load_dataset("bentrevett/multi30k")
SRC_KEY = "de"
TGT_KEY = "en"

train_m30k = multi30k["train"]
valid_m30k = multi30k["validation"]
test_m30k  = multi30k["test"]

# === (고정) Multi30k 데이터 사용 규칙 ===
train_m30k = train_m30k.select(range(min(M30K_TRAIN_SAMPLES_FIXED, len(train_m30k))))

print("Multi30k sizes:", len(train_m30k), len(valid_m30k), len(test_m30k))
print("Multi30k sample:", train_m30k[0])

# === (고정) IWSLT 데이터 로드 ===
def load_iwslt2017_de_en_parquet():
    """IWSLT 2017 (de-en) parquet 직접 로드."""
    REV = "b2dcd74c28e4544eb5c9bb76ace449ea26bbe909"
    base = f"https://huggingface.co/datasets/IWSLT/iwslt2017/resolve/{REV}/iwslt2017-de-en"

    data_files = {
        "train": f"{base}/iwslt2017-train.parquet",
        "validation": f"{base}/iwslt2017-validation.parquet",
        "test": f"{base}/iwslt2017-test.parquet",
    }
    return load_dataset("parquet", data_files=data_files)

ft_raw = load_iwslt2017_de_en_parquet()
print("IWSLT sizes:", len(ft_raw["train"]), len(ft_raw["validation"]), len(ft_raw["test"]))
print("IWSLT sample:", ft_raw["train"][0])


### STEP5. 토크나이저(BPE) 준비: Multi30k + IWSLT train 기반

- 리그전에서는 tokenizer 관련 설정을 변경하지 않습니다.
- checkpoint에 tokenizer가 있으면 그대로 불러옵니다.
- checkpoint가 없으면 Multi30k train과 IWSLT train subset을 이용해 BPE tokenizer를 학습합니다.


In [ ]:
assert "build_hf_tokenizer_from_tokenizers" in globals(), "build_hf_tokenizer_from_tokenizers가 없습니다. 위의 import/유틸(토크나이저) 셀을 먼저 실행하세요."
assert "ft_raw" in globals(), "IWSLT 원본 데이터(ft_raw)가 없습니다. STEP4 셀을 먼저 실행하세요."

import json

# === IWSLT translation field helper ===
def extract_translation_pair(trans, src_lang="de", tgt_lang="en"):
    """IWSLT translation 필드(dict 또는 JSON string)에서 src/tgt 문장을 꺼냅니다."""
    if isinstance(trans, str):
        try:
            trans = json.loads(trans)
        except Exception:
            trans = {}

    if isinstance(trans, dict):
        return trans.get(src_lang, ""), trans.get(tgt_lang, "")
    return "", ""

# === BPE tokenizer helper ===
def train_bpe_tokenizer_from_text_iterator(text_iterator_fn, vocab_size=12_000, min_freq=2):
    tok = Tokenizer(BPE(unk_token="<unk>"))
    tok.pre_tokenizer = Whitespace()
    trainer = BpeTrainer(
        vocab_size=vocab_size,
        min_frequency=min_freq,
        special_tokens=SPECIAL_TOKENS,
    )

    tok.train_from_iterator(text_iterator_fn(), trainer=trainer)

    bos_id = tok.token_to_id("<bos>")
    eos_id = tok.token_to_id("<eos>")
    tok.post_processor = TemplateProcessing(
        single="<bos> $A <eos>",
        special_tokens=[("<bos>", bos_id), ("<eos>", eos_id)],
    )
    return tok

# === checkpoint helpers ===
def checkpoint_has_tokenizer(path):
    return os.path.isdir(path) and os.path.exists(os.path.join(path, "tokenizer.json"))

def checkpoint_has_model(path):
    return (
        os.path.isdir(path)
        and os.path.exists(os.path.join(path, "config.json"))
        and (
            os.path.exists(os.path.join(path, "model.safetensors"))
            or os.path.exists(os.path.join(path, "pytorch_model.bin"))
        )
    )

PRETRAIN_CKPT_AVAILABLE = checkpoint_has_tokenizer(PRETRAIN_CKPT_DIR) and checkpoint_has_model(PRETRAIN_CKPT_DIR)

if USE_PRETRAINED_CKPT and (not FORCE_RETRAIN_PRETRAIN) and checkpoint_has_tokenizer(PRETRAIN_CKPT_DIR):
    hf_tokenizer = PreTrainedTokenizerFast.from_pretrained(PRETRAIN_CKPT_DIR)
    print("Loaded tokenizer from checkpoint:", PRETRAIN_CKPT_DIR)
else:
    iwslt_tok_train = ft_raw["train"].shuffle(seed=IWSLT_SHUFFLE_SEED_FIXED).select(
        range(min(TOKENIZER_IWSLT_SAMPLES_FIXED, len(ft_raw["train"])))
    )

    def tokenizer_text_iterator():
        # Multi30k text
        for ex in train_m30k:
            yield ex[SRC_KEY]
            yield ex[TGT_KEY]

        # IWSLT text
        for ex in iwslt_tok_train:
            src_text, tgt_text = extract_translation_pair(ex["translation"], "de", "en")
            yield src_text
            yield tgt_text

    tok = train_bpe_tokenizer_from_text_iterator(
        tokenizer_text_iterator,
        vocab_size=VOCAB_SIZE_FIXED,
        min_freq=MIN_FREQ_FIXED,
    )
    hf_tokenizer = build_hf_tokenizer_from_tokenizers(tok)
    print("Trained new tokenizer")

MAX_LEN = MAX_LEN_FIXED

print("vocab_size:", hf_tokenizer.vocab_size)
print("MAX_LEN:", MAX_LEN)
print("special tokens:", hf_tokenizer.special_tokens_map)
print("PRETRAIN_CKPT_AVAILABLE:", PRETRAIN_CKPT_AVAILABLE)


### STEP6. 공통 전처리 함수
- **목표:** HuggingFace `datasets` 형식의 예제를 모델 학습용 텐서 컬럼으로 변환합니다.
- **핵심 로직:**
  - `input_ids`: 소스 문장 토큰 ID
  - `labels`: 타깃 문장 토큰 ID (학습 시 정답)
  - `max_length`/`truncation`으로 길이를 통제합니다.
- **용어:**
  - **labels**: 분류/번역에서 정답을 의미. Seq2Seq에서는 '다음 토큰' 정답열입니다.
  - **ignore_index(-100)**: loss 계산에서 무시할 위치(보통 padding)를 표시하는 값(HF Trainer가 사용).

In [ ]:
# === 공통 전처리 함수 ===
def preprocess_m30k(examples):
    model_inputs = hf_tokenizer(examples[SRC_KEY], max_length=MAX_LEN, truncation=True)
    labels = hf_tokenizer(text_target=examples[TGT_KEY], max_length=MAX_LEN, truncation=True)
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

train_m30k_tok = train_m30k.map(preprocess_m30k, batched=True, remove_columns=train_m30k.column_names)
valid_m30k_tok = valid_m30k.map(preprocess_m30k, batched=True, remove_columns=valid_m30k.column_names)
test_m30k_tok  = test_m30k.map(preprocess_m30k,  batched=True, remove_columns=test_m30k.column_names)

train_m30k_tok[0]

### STEP7. 고정 Transformer 모델 / checkpoint 로드

- 리그전에서는 모델 구조를 변경하지 않습니다.
- checkpoint가 있으면 고정 pre-train 모델을 로드합니다.
- checkpoint가 없으면 고정 구조로 모델을 만들고 Stage A에서 pre-train합니다.


In [ ]:
# === Fixed model / checkpoint 로드 ===
# MODEL_HP는 STEP3-1에서 고정된 값으로 설정됩니다.

print("Fixed MODEL_HP:", MODEL_HP)

def build_model(d_model, nhead, num_layers, d_ff, dropout):
    assert d_model % nhead == 0, f"d_model({d_model}) must be divisible by nhead({nhead})"

    config = BartConfig(
        vocab_size=hf_tokenizer.vocab_size,
        d_model=d_model,
        encoder_layers=num_layers,
        decoder_layers=num_layers,
        encoder_attention_heads=nhead,
        decoder_attention_heads=nhead,
        encoder_ffn_dim=d_ff,
        decoder_ffn_dim=d_ff,
        dropout=dropout,
        attention_dropout=dropout,
        activation_dropout=dropout,
        max_position_embeddings=MAX_LEN + 2,
        pad_token_id=hf_tokenizer.pad_token_id,
        bos_token_id=hf_tokenizer.bos_token_id,
        eos_token_id=hf_tokenizer.eos_token_id,
        decoder_start_token_id=hf_tokenizer.bos_token_id,
        attn_implementation="eager",
    )
    return BartForConditionalGeneration(config)

PRETRAIN_CKPT_AVAILABLE = checkpoint_has_tokenizer(PRETRAIN_CKPT_DIR) and checkpoint_has_model(PRETRAIN_CKPT_DIR)

if USE_PRETRAINED_CKPT and (not FORCE_RETRAIN_PRETRAIN) and PRETRAIN_CKPT_AVAILABLE:
    model = BartForConditionalGeneration.from_pretrained(PRETRAIN_CKPT_DIR).to(device)
    PRETRAIN_CKPT_LOADED = True
    print("Loaded pre-trained model from checkpoint:", PRETRAIN_CKPT_DIR)
else:
    model = build_model(**MODEL_HP).to(device)
    PRETRAIN_CKPT_LOADED = False
    print("Created new fixed model. Stage A pre-train will run.")

if model.config.vocab_size != hf_tokenizer.vocab_size:
    raise ValueError(f"model vocab_size({model.config.vocab_size}) != tokenizer vocab_size({hf_tokenizer.vocab_size})")

print("model params (M):", sum(p.numel() for p in model.parameters()) / 1e6)


### STEP8. BLEU metric
- **목표:** 번역 품질을 BLEU로 평가하는 함수를 정의합니다.
- **BLEU가 뭐예요?**
  - 모델 번역(hypothesis)과 정답(reference) 사이의 n-gram 겹침을 기반으로 하는 점수입니다.
  - 높을수록 좋지만, 사람 평가와 100% 일치하진 않습니다(그래도 리그전 지표로 많이 씀).
- **주의:**
  - 토크나이징 방식(BPE/공백), 특수토큰 제거 여부에 따라 BLEU가 달라질 수 있어요.

In [ ]:
import re

_detok_punct_re = re.compile(r"\s+([?.!,;:])")
_detok_contraction_re = re.compile(r"\s+(n't|'re|'ve|'ll|'d|'m|'s)\b")

def simple_detokenize(text: str) -> str:
    text = text.strip()
    text = _detok_punct_re.sub(r"\1", text)
    text = _detok_contraction_re.sub(r"\1", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def compute_metrics(eval_pred):
    # transformers 버전에 따라 eval_pred가 (preds, labels) 튜플이거나 EvalPrediction일 수 있음
    preds = eval_pred.predictions if hasattr(eval_pred, "predictions") else eval_pred[0]
    labels = eval_pred.label_ids if hasattr(eval_pred, "label_ids") else eval_pred[1]

    # tuple(preds, ..) 형태로 오는 경우가 있음
    if isinstance(preds, tuple):
        preds = preds[0]

    pad_id = hf_tokenizer.pad_token_id
    unk_id = hf_tokenizer.unk_token_id if hf_tokenizer.unk_token_id is not None else pad_id

    # ✅ v5: -100 padding 제거 (중요!)
    preds  = np.where(preds  != -100, preds,  pad_id)
    labels = np.where(labels != -100, labels, pad_id)

    # 추가 안전장치: vocab 범위를 벗어나면 unk로 치환
    preds  = np.where((preds  < 0) | (preds  >= hf_tokenizer.vocab_size), unk_id, preds)
    labels = np.where((labels < 0) | (labels >= hf_tokenizer.vocab_size), unk_id, labels)

    pred_str  = hf_tokenizer.batch_decode(preds,  skip_special_tokens=True)
    label_str = hf_tokenizer.batch_decode(labels, skip_special_tokens=True)

    # ✅ detokenize + strip
    pred_str  = [simple_detokenize(p) for p in pred_str]
    label_str = [simple_detokenize(l) for l in label_str]

    # force=True: "토큰화된 것 같다"는 경고를 억제(우리는 위에서 detokenize도 수행)
    bleu = sacrebleu.corpus_bleu(pred_str, [label_str], force=True).score
    return {"bleu": bleu}


### STEP9. Stage A: Pre-train 또는 checkpoint 사용

- checkpoint가 있으면 Stage A를 건너뜁니다.
- checkpoint가 없으면 고정된 모델 구조와 학습 설정으로 Multi30k pre-train을 수행한 뒤 모델과 tokenizer를 저장합니다.
- 이후 `FT_HP`만 바꿔 다시 실행할 때는 같은 pre-train checkpoint에서 fine-tune을 다시 시작합니다.


In [ ]:
# === Stage A: Pre-train on Multi30k 또는 checkpoint 사용 ===
from datetime import datetime

if PRETRAIN_CKPT_LOADED:
    print("Stage A skipped. Using cached pre-trained checkpoint:", PRETRAIN_CKPT_DIR)
    pre_train_result = None
    pre_log_history = []
    pre_valid = {"eval_bleu": None, "loaded_checkpoint": PRETRAIN_CKPT_DIR}
else:
    PRETRAIN_RUN_TAG = datetime.now().strftime("%Y%m%d_%H%M%S")
    PRETRAIN_OUT_DIR = f"./runs/league_pretrain_multi30k_fixed_{PRETRAIN_RUN_TAG}"

    pre_args = make_seq2seq_training_args(
        output_dir=PRETRAIN_OUT_DIR,
        max_steps=PRETRAIN_MAX_STEPS_FIXED,
        logging_steps=50,
        eval_steps=PRETRAIN_EVAL_STEPS_FIXED,
        eval_strategy="steps",

        save_steps=PRETRAIN_EVAL_STEPS_FIXED,
        save_strategy="steps",
        save_total_limit=2,
        load_best_model_at_end=True,
        metric_for_best_model="eval_bleu",
        greater_is_better=True,

        per_device_train_batch_size=PRETRAIN_BSZ_FIXED,
        per_device_eval_batch_size=PRETRAIN_BSZ_FIXED,
        gradient_accumulation_steps=PRETRAIN_ACCUM_FIXED,

        learning_rate=PRETRAIN_LR_FIXED,
        lr_scheduler_type=PRETRAIN_SCHEDULER_FIXED,
        warmup_steps=PRETRAIN_WARMUP_FIXED,

        predict_with_generate=True,
        generation_max_length=MAX_LEN,
        generation_num_beams=PRETRAIN_GEN_BEAMS_FIXED,

        fp16=torch.cuda.is_available(),
        report_to=[],
    )

    collator = DataCollatorForSeq2Seq(hf_tokenizer, model=model)

    try:
        pre_trainer = Seq2SeqTrainer(
            model=model,
            args=pre_args,
            train_dataset=train_m30k_tok,
            eval_dataset=valid_m30k_tok,
            processing_class=hf_tokenizer,
            data_collator=collator,
            compute_metrics=compute_metrics,
        )
    except TypeError:
        pre_trainer = Seq2SeqTrainer(
            model=model,
            args=pre_args,
            train_dataset=train_m30k_tok,
            eval_dataset=valid_m30k_tok,
            tokenizer=hf_tokenizer,
            data_collator=collator,
            compute_metrics=compute_metrics,
        )

    pre_train_result = pre_trainer.train()
    pre_valid = pre_trainer.evaluate()
    print("pre_valid:", pre_valid)

    pre_log_history = pre_trainer.state.log_history
    model = pre_trainer.model

    os.makedirs(PRETRAIN_CKPT_DIR, exist_ok=True)
    model.save_pretrained(PRETRAIN_CKPT_DIR)
    hf_tokenizer.save_pretrained(PRETRAIN_CKPT_DIR)
    print("Saved pre-trained checkpoint:", PRETRAIN_CKPT_DIR)

stage1_eval_after = pre_valid


### STEP10. Stage B: Fine-tune 데이터 준비

- IWSLT train/validation/test 데이터를 fine-tune 및 평가용으로 준비합니다.
- `public_fast`는 train subset과 validation subset으로 빠르게 비교합니다.
- `public_full`은 전체 validation으로 제출 전 확인합니다.
- `final`은 IWSLT test BLEU까지 계산합니다.


In [ ]:
# === (고정) Fine-tune 데이터 사용 규칙 ===
# IWSLT 원본 데이터(ft_raw)는 STEP4에서 이미 로드되었습니다.
print("IWSLT train samples for this mode:", IWSLT_TRAIN_SAMPLES_FIXED)
print("IWSLT shuffle seed:", IWSLT_SHUFFLE_SEED_FIXED)
print("RUN_TEST_EVAL:", RUN_TEST_EVAL)
ft_raw


### STEP11. Fine-tune 전처리

- IWSLT `translation` 필드를 tokenizer 입력/label로 변환합니다.
- `final` 모드일 때만 test set을 tokenizing하여 최종 test BLEU를 계산합니다.


In [ ]:
# === Fine-tune 전처리 (IWSLT 고정) ===
def preprocess_translation_dict(examples, src_lang="de", tgt_lang="en"):
    """translation 필드(dict 또는 JSON string)를 (input_ids, labels)로 변환합니다."""
    trans = examples["translation"]

    src_texts, tgt_texts = [], []
    for ex in trans:
        src_text, tgt_text = extract_translation_pair(ex, src_lang, tgt_lang)
        src_texts.append(src_text)
        tgt_texts.append(tgt_text)

    model_inputs = hf_tokenizer(src_texts, max_length=MAX_LEN, truncation=True)

    try:
        labels = hf_tokenizer(text_target=tgt_texts, max_length=MAX_LEN, truncation=True)
    except TypeError:
        with hf_tokenizer.as_target_tokenizer():
            labels = hf_tokenizer(tgt_texts, max_length=MAX_LEN, truncation=True)

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

# IWSLT train은 LEAGUE_MODE에 따라 subset 크기가 달라집니다.
ft_train = ft_raw["train"].shuffle(seed=IWSLT_SHUFFLE_SEED_FIXED).select(
    range(min(IWSLT_TRAIN_SAMPLES_FIXED, len(ft_raw["train"])))
)
ft_valid = ft_raw["validation"]
ft_test  = ft_raw["test"]

ft_train_tok = ft_train.map(lambda x: preprocess_translation_dict(x, "de", "en"), batched=True, remove_columns=ft_train.column_names)
ft_valid_tok = ft_valid.map(lambda x: preprocess_translation_dict(x, "de", "en"), batched=True, remove_columns=ft_valid.column_names)

if VALID_EVAL_SAMPLES_FIXED is None:
    ft_valid_eval_tok = ft_valid_tok
else:
    ft_valid_eval_tok = ft_valid_tok.select(range(min(VALID_EVAL_SAMPLES_FIXED, len(ft_valid_tok))))

if RUN_TEST_EVAL:
    ft_test_tok = ft_test.map(lambda x: preprocess_translation_dict(x, "de", "en"), batched=True, remove_columns=ft_test.column_names)
else:
    ft_test_tok = None

print("IWSLT train samples:", len(ft_train_tok))
print("IWSLT valid eval samples:", len(ft_valid_eval_tok), "/", len(ft_valid_tok))
print("IWSLT test eval enabled:", RUN_TEST_EVAL)


### STEP12. Stage B: Fine-tune (튜닝 구간)

- **목표:** IWSLT로 fine-tune을 수행한 뒤 validation BLEU 또는 test BLEU를 출력합니다.
- **리그전 튜닝:** `FT_HP` 값만 조정합니다.
- **고정:** pre-train 모델 구조, pre-train 학습 설정, tokenizer, 데이터 전처리, batch size, step budget은 고정합니다.


In [ ]:
# === Stage B: Fine-tune (🔧 TUNE: optimizer/schedule) ===
# 리그전에서 바꿔도 되는 값: Fine-tune Optimizer / Schedule 하이퍼파라미터

FT_HP = dict(
    learning_rate=1e-4,            # 🔧 TUNE: 1e-5 ~ 5e-4
    lr_scheduler_type="linear",    # 🔧 TUNE: linear / cosine / inverse_sqrt
    warmup_steps=50,               # 🔧 TUNE: 0 ~ 300
    weight_decay=0.0,              # 🔧 TUNE: 0.0 ~ 0.1
    label_smoothing_factor=0.10,   # 🔧 TUNE: 0.0 ~ 0.20
)

def validate_ft_hp(ft_hp):
    required_keys = {
        "learning_rate",
        "lr_scheduler_type",
        "warmup_steps",
        "weight_decay",
        "label_smoothing_factor",
    }
    if set(ft_hp.keys()) != required_keys:
        raise ValueError(f"FT_HP keys must be exactly {required_keys}")

    lr = float(ft_hp["learning_rate"])
    warmup = int(ft_hp["warmup_steps"])
    wd = float(ft_hp["weight_decay"])
    smoothing = float(ft_hp["label_smoothing_factor"])

    if not (1e-5 <= lr <= 5e-4):
        raise ValueError("learning_rate must be between 1e-5 and 5e-4")
    if ft_hp["lr_scheduler_type"] not in {"linear", "cosine", "inverse_sqrt"}:
        raise ValueError("lr_scheduler_type must be one of: linear, cosine, inverse_sqrt")
    if not (0 <= warmup <= 300):
        raise ValueError("warmup_steps must be between 0 and 300")
    if not (0.0 <= wd <= 0.1):
        raise ValueError("weight_decay must be between 0.0 and 0.1")
    if not (0.0 <= smoothing <= 0.2):
        raise ValueError("label_smoothing_factor must be between 0.0 and 0.2")

validate_ft_hp(FT_HP)

from datetime import datetime
FT_RUN_TAG = datetime.now().strftime("%Y%m%d_%H%M%S")
FT_OUT_DIR = f"./runs/league_finetune_iwslt_{LEAGUE_MODE}_{PRETRAIN_MODEL_TAG}_{FT_RUN_TAG}"

# FT_HP 실험을 공정하게 비교하기 위해 fine-tune 시작 전 같은 base checkpoint를 다시 로드합니다.
if checkpoint_has_model(PRETRAIN_CKPT_DIR) and checkpoint_has_tokenizer(PRETRAIN_CKPT_DIR):
    model = BartForConditionalGeneration.from_pretrained(PRETRAIN_CKPT_DIR).to(device)
    print("Fine-tune start checkpoint:", PRETRAIN_CKPT_DIR)
else:
    print("Pre-train checkpoint not found. Using current model state.")
    model = model.to(device)

ft_args = make_seq2seq_training_args(
    output_dir=FT_OUT_DIR,
    max_steps=FINETUNE_MAX_STEPS_FIXED,
    logging_steps=50,
    eval_steps=FINETUNE_EVAL_STEPS_FIXED,
    eval_strategy="steps",

    save_steps=FINETUNE_EVAL_STEPS_FIXED,
    save_strategy="steps",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_bleu",
    greater_is_better=True,

    per_device_train_batch_size=FINETUNE_BSZ_FIXED,
    per_device_eval_batch_size=FINETUNE_BSZ_FIXED,
    gradient_accumulation_steps=FINETUNE_ACCUM_FIXED,

    learning_rate=FT_HP["learning_rate"],
    lr_scheduler_type=FT_HP["lr_scheduler_type"],
    warmup_steps=FT_HP["warmup_steps"],
    weight_decay=FT_HP["weight_decay"],
    label_smoothing_factor=FT_HP["label_smoothing_factor"],

    predict_with_generate=True,
    generation_max_length=MAX_LEN,
    generation_num_beams=GEN_BEAMS_FIXED,

    fp16=torch.cuda.is_available(),
    report_to=[],
)

try:
    ft_trainer = Seq2SeqTrainer(
        model=model,
        args=ft_args,
        train_dataset=ft_train_tok,
        eval_dataset=ft_valid_eval_tok,
        processing_class=hf_tokenizer,
        data_collator=DataCollatorForSeq2Seq(hf_tokenizer, model=model),
        compute_metrics=compute_metrics,
    )
except TypeError:
    ft_trainer = Seq2SeqTrainer(
        model=model,
        args=ft_args,
        train_dataset=ft_train_tok,
        eval_dataset=ft_valid_eval_tok,
        tokenizer=hf_tokenizer,
        data_collator=DataCollatorForSeq2Seq(hf_tokenizer, model=model),
        compute_metrics=compute_metrics,
    )

ft_train_result = ft_trainer.train()

# Validation score
ft_iwslt_valid = ft_trainer.evaluate(eval_dataset=ft_valid_eval_tok, metric_key_prefix="iwslt_valid")
VALID_BLEU = ft_iwslt_valid.get("iwslt_valid_bleu")

# Final test score: final 모드에서만 계산합니다.
if RUN_TEST_EVAL:
    assert ft_test_tok is not None, "RUN_TEST_EVAL=True이면 ft_test_tok가 준비되어 있어야 합니다."
    ft_iwslt_test = ft_trainer.evaluate(eval_dataset=ft_test_tok, metric_key_prefix="iwslt_test")
    FINAL_TEST_BLEU = ft_iwslt_test.get("iwslt_test_bleu")
else:
    ft_iwslt_test = None
    FINAL_TEST_BLEU = None

print("Fine-tune train result:", ft_train_result)
print("VALID_BLEU:", VALID_BLEU)
print("LEAGUE_VALID_SCORE_IWSLT_VALID_BLEU:", VALID_BLEU)

if FINAL_TEST_BLEU is not None:
    print("FINAL_TEST_BLEU:", FINAL_TEST_BLEU)
    print("\n=== LEAGUE FINAL SCORE ===")
    print("LEAGUE_FINAL_SCORE_IWSLT_TEST_BLEU:", FINAL_TEST_BLEU)
else:
    print("\n=== VALIDATION SCORE ONLY ===")
    print("Set LEAGUE_MODE = 'final' to compute IWSLT test BLEU.")

run_summary = {
    "league_mode": LEAGUE_MODE,
    "ft_hp": FT_HP,
    "valid_bleu": VALID_BLEU,
    "final_iwslt_test_bleu": FINAL_TEST_BLEU,
    "fixed": {
        "model_hp": MODEL_HP,
        "max_len": MAX_LEN,
        "vocab_size": VOCAB_SIZE_FIXED,
        "pretrain_checkpoint": PRETRAIN_CKPT_DIR,
        "iwslt_train_samples": IWSLT_TRAIN_SAMPLES_FIXED,
        "finetune_max_steps": FINETUNE_MAX_STEPS_FIXED,
        "gen_beams": GEN_BEAMS_FIXED,
    },
}

print("\n=== RUN SUMMARY (copy-paste) ===")
print(json.dumps(run_summary, ensure_ascii=False, indent=2))

ft_log_history = ft_trainer.state.log_history


---
## 📌 제작 정보 & 출처

- 제작: 가천대학교 인공지능 학술 동아리 **Aing (A.ing)**

### 사용/참고 자료
- Vaswani et al., **Attention Is All You Need**, NeurIPS 2017.
- A.ing 내부 스터디 자료: *Attention 치트시트*, *Transformer CookBook*.
- HuggingFace Transformers/Datasets 문서: Seq2Seq 학습(Trainer), Multi30k, IWSLT 로딩.
- `tokenizers`(BPE), `sacrebleu`(BLEU 평가) 라이브러리.
